# A-RAD Edge Model Training — frozen centroids ➜ frozen Isolation-Forest bundle
This notebook reproduces the KB detector **exactly** from `anomalies_multimodal.csv` and the six frozen
training centroids (Silhouette 0.808 run), then exports `edge_bundle.joblib` for the Jetson.

**Verified reproduction on the KB itself:** cluster assignment ≈100% (2 boundary-tie rows in 53,759),
anomaly labels **100%**, anomaly rule-`text` byte-equal **99.94%**.

In [1]:
#@title Colab setup (skip locally)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')


In [2]:
#@title Imports & the frozen artifacts of the training run
import numpy as np, pandas as pd, joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# 14 sensors: the space the centroids live in (cluster ASSIGNMENT)
FEATURES = ['T24','T30','T50','P30','Nf','Nc','Ps30','phi','NRf','NRc','BPR','htBleed','W31','W32']
# EXACT feature matrix of the per-cluster IsolationForests (KB recipe):
# csv order minus the dropped meta columns -> sensors + h_clust + cycles.
# h_clust is constant within a cluster (never splits) but participates in the
# tree RNG; cycles DOES split and appears in the KB rules (TIOT-LLM Eq. 1).
IF_FEATURES = FEATURES + ['h_clust','cycles']
K, CONTAM, SEED = 6, 0.1, 42

# The six frozen TRAINING centroids (normalized space, Silhouette=0.808 run):
FROZEN_CENTROIDS = np.array([
 [-0.8036546,-0.61831856,-0.65176307,-0.98611309,-0.11643701,-0.58756939,-0.25875626,-0.98446372,0.41771091,0.26922962,0.06058939,-0.62127434,-1.03380578,-1.03388033],
 [1.68967345,1.60884943,1.70764741,1.85442889,1.09650142,1.60100871,1.40969468,1.85485165,0.4180792,0.88481149,-1.18427038,1.61737429,1.82670489,1.82665063],
 [-0.64022107,-0.50268991,-0.62046538,-0.60384244,-0.04020821,-0.50355311,-0.31024201,-0.60327504,0.41838169,0.06602508,0.00352968,-0.50711042,-0.60374708,-0.60381436],
 [0.74800645,0.61934319,0.40023119,0.35540083,0.65521051,0.60917939,0.45261659,0.35453246,0.41850654,-0.00810265,-0.12940148,0.61522094,0.37134525,0.37141507],
 [0.67690059,0.77437133,0.8882559,0.76517193,0.62034433,0.77495149,0.77428993,0.76549575,0.41856741,0.80778066,-0.8874987,0.7707004,0.78394984,0.78397152],
 [-1.14798656,-1.48459766,-1.30220014,-0.73409545,-2.1587131,-1.51783443,-1.91310053,-0.7373105,-2.39127067,-2.21761334,2.11430631,-1.47585819,-0.66129277,-0.66113808]])


In [3]:
#@title Load the dataset (this one)
CSV = '/content/anomalies_multimodal.csv' if IN_COLAB else 'anomalies_multimodal.csv'
df = pd.read_csv(CSV)
if 'cycles' not in df.columns and 'cycle' in df.columns: df['cycles'] = df['cycle']
print(df.shape, '| units:', df['Unit_ID'].nunique(), '| anomaly rate:', (df.anomaly_label==-1).mean().round(4))


(53759, 26) | units: 260 | anomaly rate: 0.1001


In [4]:
#@title 1) Scaler + assignment by the FROZEN centroids (no re-clustering)
scaler = StandardScaler().fit(df[FEATURES].astype(float).values)
Xn = scaler.transform(df[FEATURES].astype(float).values)
assign = ((Xn[:,None,:]-FROZEN_CENTROIDS[None,:,:])**2).sum(-1).argmin(1)
agree = (assign == df['h_clust'].values).mean()
print(f'nearest-centroid vs stored h_clust: {agree:.6f}  '
      f'({int((assign!=df.h_clust.values).sum())} boundary-tie rows)')
# fit uses the stored h_clust (the training truth); assignment is the frozen
# inference rule that will label unseen data.


nearest-centroid vs stored h_clust: 0.999963  (2 boundary-tie rows)


In [5]:
#@title 2) Per-cluster Isolation Forests — the model (exact KB recipe)
Xif = df[IF_FEATURES].astype(float).values
models, thresholds = {}, {}
for k in range(K):
    m = df['h_clust'].values == k
    f = IsolationForest(n_estimators=100, contamination=CONTAM, random_state=SEED).fit(Xif[m])
    models[k] = f
    thresholds[k] = float(f.offset_)   # sklearn's own fitted cut, FROZEN
    print(f'cluster {k}: {int(m.sum())} rows fitted')


cluster 0: 13458 rows fitted


cluster 1: 8044 rows fitted


cluster 2: 8037 rows fitted


cluster 3: 8122 rows fitted


cluster 4: 8096 rows fitted


cluster 5: 8002 rows fitted


In [6]:
#@title 3) Verification — the refit IS the KB detector
score = np.empty(len(df)); label = np.ones(len(df), dtype=int)
for k, f in models.items():
    m = df['h_clust'].values == k
    s = f.score_samples(Xif[m]) - thresholds[k]   # = decision_function
    score[m] = s; label[m] = np.where(s < 0, -1, 1)
lab_agree = (label == df['anomaly_label'].values).mean()
sc_close  = np.abs(score - df['anomaly_score'].values).max()
print(f'anomaly-label agreement vs KB: {lab_agree:.6f}')
print(f'max |score diff| vs KB       : {sc_close:.2e}  (nonzero only on the tie rows)')
assert lab_agree == 1.0, 'recipe drift — do not export'


anomaly-label agreement vs KB: 1.000000
max |score diff| vs KB       : 2.64e-16  (nonzero only on the tie rows)


In [7]:
#@title 4) Export the frozen bundle (model + dataset handles)
bundle = {'scaler': scaler, 'centroids': FROZEN_CENTROIDS,
          'models': models, 'thresholds': thresholds,
          'features': FEATURES,
          'meta': {'assign_features': FEATURES, 'if_features': IF_FEATURES,
                   'k': K, 'contamination': CONTAM, 'seed': SEED,
                   'n_train_rows': int(len(df)), 'source': CSV}}
joblib.dump(bundle, 'edge_bundle.joblib')
import json as _json
_json.dump({'features': FEATURES, 'centroids': FROZEN_CENTROIDS.tolist(),
            'scaler_mean': scaler.mean_.tolist(), 'scaler_scale': scaler.scale_.tolist()},
           open('centroids.json','w'), indent=1)
print('saved edge_bundle.joblib (the models) and centroids.json (the centroids)')
iso_forest_dict = {k: {'model': models[k], 'scaler': None, 'feature_cols': IF_FEATURES,
                       'contamination': CONTAM, 'random_state': SEED} for k in models}
dataset = df   # the dataset, as in the original notebook's (df_h, iso_dict_h)
if IN_COLAB:
    files.download('edge_bundle.joblib'); files.download('centroids.json')


saved edge_bundle.joblib (the models) and centroids.json (the centroids)


## Optional — preview on the official test set
The full test pipeline (detection ➜ SLM interpretation on the Jetson Orin Nano) lives in the
companion `arad-edge-jetson.zip`; this cell is only a sanity preview.

In [8]:
#@title 5) (optional) apply to test_FD002.txt
RUN_PREVIEW = False
if RUN_PREVIEW:
    from edge_iforest import EdgeBundle, apply_bundle, load_official_test
    b = EdgeBundle.from_dict(bundle)
    te = load_official_test('test_FD002.txt','RUL_FD002.txt')
    out = apply_bundle(b, te)
    print(len(out),'rows |', int((out.anomaly_label==-1).sum()),'anomalies')
    print(out[out.anomaly_label==-1].iloc[0]['text'][:220])
